In [11]:
import numpy as np
from gensim.models import KeyedVectors
import os

# Just to see where the notebook is running
print("Current working directory:", os.getcwd())


Current working directory: c:\Users\Asus\OneDrive\Desktop\word translation


In [12]:
from gensim.models import KeyedVectors

CS_VEC_FILE = "cc.cs.300.vec.gz"
EN_VEC_FILE = "cc.en.300.vec.gz"

print("Loading Czech embeddings...")
cs_vecs = KeyedVectors.load_word2vec_format(CS_VEC_FILE, binary=False)
print("Czech embeddings loaded. Vocab size:", len(cs_vecs))

print("Loading English embeddings...")
en_vecs = KeyedVectors.load_word2vec_format(EN_VEC_FILE, binary=False)
print("English embeddings loaded. Vocab size:", len(en_vecs))


Loading Czech embeddings...
Czech embeddings loaded. Vocab size: 2000000
Loading English embeddings...
English embeddings loaded. Vocab size: 2000000


In [ ]:
def load_dictionary(path):
    pairs = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) == 2:
                pairs.append((parts[0], parts[1]))
    return pairs

train_pairs = load_dictionary("cs-en.0-5000.txt") # frequent words common -> more pairs
test_pairs  = load_dictionary("cs-en.5000-6500.txt") #less frequent words

print("Train pairs:", len(train_pairs))
print("Test pairs:", len(test_pairs))

print("First 10 train pairs:")
for i in range(10):
    print(train_pairs[i])


Train pairs: 7349
Test pairs: 2071
First 10 train pairs:
('byl', 'was')
('jako', 'as')
('jako', 'like')
('roce', 'year')
('pro', 'for')
('roku', 'roku')
('roku', 'year')
('jsou', 'are')
('jeho', 'his')
('jeho', 'its')


In [ ]:
import numpy as np

def build_xy(pairs, cs_vecs, en_vecs):
    X, Y = [], []
    skipped = 0
    for cs, en in pairs:
        if cs in cs_vecs and en in en_vecs:
            X.append(cs_vecs[cs])
            Y.append(en_vecs[en])
        else:
            skipped += 1 
    return np.array(X), np.array(Y), skipped

X_train, Y_train, skipped_train = build_xy(train_pairs, cs_vecs, en_vecs)
X_test,  Y_test,  skipped_test  = build_xy(test_pairs,  cs_vecs, en_vecs)

print("X_train:", X_train.shape)
print("Y_train:", Y_train.shape)
print("Skipped train:", skipped_train)

print("X_test:", X_test.shape)
print("Y_test:", Y_test.shape)
print("Skipped test:", skipped_test)


X_train: (7125, 300)
Y_train: (7125, 300)
Skipped train: 224
X_test: (1963, 300)
Y_test: (1963, 300)
Skipped test: 108


In [ ]:
import numpy as np

# ---------------------------------------------------------
# 1. Frobenius Norm  ||A||_F^2
# ---------------------------------------------------------
def frobenius_norm_squared(A):
    return np.sum(A * A)


# ---------------------------------------------------------
# 2. Compute XWᵀ - Y
# ---------------------------------------------------------
def compute_difference(X, W, Y):
    return X @ W.T - Y


# ---------------------------------------------------------
# 3. Gradient of Loss wrt Wᵀ
# ---------------------------------------------------------
def compute_gradient(X, W, Y):
    E = compute_difference(X, W, Y)
    grad = 2 * (E.T @ X)
    return grad


# ---------------------------------------------------------
# 4. One Gradient Descent Step
# ---------------------------------------------------------
def gradient_descent_step(X, W, Y, alpha):
    grad = compute_gradient(X, W, Y)
    return W - alpha * grad


# ---------------------------------------------------------
# 5. Training Loop with Convergence
# ---------------------------------------------------------
def train_mapping(X, Y, steps=300, alpha=1e-5):
    d = X.shape[1]
    W = np.eye(d)
    losses = []

    print("====================================")
    print("      TRAINING LINEAR MAPPING")
    print("====================================")
    print(f"Embedding dimension: {d}")
    print(f"Learning rate: {alpha}")
    print(f"Max steps: {steps}")
    print("------------------------------------")

    for step in range(steps):
        diff = compute_difference(X, W, Y)
        loss = frobenius_norm_squared(diff)
        losses.append(loss)

        # Print every 20 steps
        if step % 20 == 0:
            print(f"Step {step:4d} | Loss = {loss:.4f}")

        # Update W
        W = gradient_descent_step(X, W, Y, alpha)

        # Early stopping: if loss stops improving
        if len(losses) > 10:
            if losses[-1] > min(losses[-10:]):
                print("------------------------------------")
                print(f"Early stopping at step {step}")
                print("------------------------------------")
                break

    print("====================================")
    print(f"Training finished. Final loss: {losses[-1]:.4f}")
    print("====================================")

    return W, losses
W, losses = train_mapping(X_train, Y_train, steps=300, alpha=1e-5)


In [ ]:
print("First 10 losses:", losses[:10])


First 10 losses: [16057.208912935866, 15983.775181051007, 15912.27903093526, 15842.65558468236, 15774.842217985419, 15708.778481253781, 15644.40602349646, 15581.668518874816, 15520.511595831083, 15460.882768702024]


In [17]:

import random

cs_word, en_true = random.choice(train_pairs)
print("\nCzech word:", cs_word)
print("True English:", en_true)

x = cs_vecs[cs_word]
x_mapped = x @ W.T

best_word = None
best_sim = -1

for en_word in en_vecs.key_to_index.keys():
    sim = np.dot(x_mapped, en_vecs[en_word]) / (
        np.linalg.norm(x_mapped) * np.linalg.norm(en_vecs[en_word]) + 1e-9
    )
    if sim > best_sim:
        best_sim = sim
        best_word = en_word

print("Predicted English:", best_word)
print("Similarity:", best_sim)



Czech word: opravy
True English: repairs
Predicted English: technology-based
Similarity: 0.33270985167323175


In [18]:
def top_k(cs_word, k=5):
    x = cs_vecs[cs_word]
    x_mapped = x @ W.T

    sims = []
    for en_word in en_vecs.key_to_index.keys():
        sim = np.dot(x_mapped, en_vecs[en_word]) / (
            np.linalg.norm(x_mapped) * np.linalg.norm(en_vecs[en_word]) + 1e-9
        )
        sims.append((en_word, sim))

    sims.sort(key=lambda t: t[1], reverse=True)
    return sims[:k]

print("\nTop-5 predictions:")
for w, s in top_k(cs_word):
    print(f"{w:15s} | sim = {s:.4f}")



Top-5 predictions:
technology-based | sim = 0.3327
designs         | sim = 0.3270
lay-up          | sim = 0.3208
MGU-H           | sim = 0.3204
MP4-24          | sim = 0.3200


In [24]:
# Reduce English vocabulary to top 10k most frequent words
MAX_VOCAB = 10000

en_words = list(en_vecs.key_to_index.keys())[:MAX_VOCAB]
en_matrix = en_vecs.vectors[:MAX_VOCAB]

print("Reduced English vocab size:", len(en_words))


Reduced English vocab size: 10000


In [25]:
def translate_word(cs_word, W, cs_vecs, en_words, en_matrix, top_k=5):
    if cs_word not in cs_vecs:
        print("Word not found:", cs_word)
        return []

    x = cs_vecs[cs_word]
    x_mapped = x @ W.T
    x_mapped = x_mapped / (np.linalg.norm(x_mapped) + 1e-9)

    # Normalize English matrix
    en_norms = np.linalg.norm(en_matrix, axis=1, keepdims=True)
    en_matrix_norm = en_matrix / (en_norms + 1e-9)

    # Cosine similarity
    sims = en_matrix_norm @ x_mapped

    # Top-k
    top_idx = np.argpartition(sims, -top_k)[-top_k:]
    top_idx = top_idx[np.argsort(sims[top_idx])[::-1]]

    return [(en_words[i], sims[i]) for i in top_idx]


In [26]:
cs_word = "rok"
results = translate_word(cs_word, W, cs_vecs, en_words, en_matrix)

print("Czech word:", cs_word)
print("Top-5 English predictions:")
for w, s in results:
    print(f"{w:15s} | sim = {s:.4f}")


Czech word: rok
Top-5 English predictions:
day             | sim = 0.3739
time            | sim = 0.3322
every           | sim = 0.3315
month           | sim = 0.3237
minute          | sim = 0.3216


In [29]:
def evaluate_accuracy(test_pairs, X_test, W, en_words, en_matrix, top_k=5):
    # Normalize English matrix
    en_norms = np.linalg.norm(en_matrix, axis=1, keepdims=True)
    en_matrix_norm = en_matrix / (en_norms + 1e-9)

    # Map Czech test vectors
    X_mapped = X_test @ W.T
    X_norms = np.linalg.norm(X_mapped, axis=1, keepdims=True)
    X_mapped_norm = X_mapped / (X_norms + 1e-9)

    # Cosine similarity matrix
    similarity_matrix = X_mapped_norm @ en_matrix_norm.T

    top1_correct = 0
    top5_correct = 0
    total = len(test_pairs)

    for i, (cs_word, en_true) in enumerate(test_pairs):
        sims = similarity_matrix[i]

        # Top-1
        best_idx = np.argmax(sims)
        if en_words[best_idx] == en_true:
            top1_correct += 1

        # Top-5
        top5_idx = np.argpartition(sims, -top_k)[-top_k:]
        top5_words = [en_words[j] for j in top5_idx]
        if en_true in top5_words:
            top5_correct += 1

    return top1_correct / total, top5_correct / total


In [31]:
# Build aligned test pairs (only those that exist in embeddings)
aligned_test_pairs = []
for cs, en in test_pairs:
    if cs in cs_vecs and en in en_vecs:
        aligned_test_pairs.append((cs, en))

print("Original test pairs:", len(test_pairs))
print("Aligned test pairs:", len(aligned_test_pairs))
print("X_test rows:", len(X_test))


Original test pairs: 2071
Aligned test pairs: 1963
X_test rows: 1963


In [32]:
top1, top5 = evaluate_accuracy(aligned_test_pairs, X_test, W, en_words, en_matrix)

print("====================================")
print("        TRANSLATION ACCURACY")
print("====================================")
print(f"Top-1 Accuracy: {top1:.4f}")
print(f"Top-5 Accuracy: {top5:.4f}")
print("====================================")


        TRANSLATION ACCURACY
Top-1 Accuracy: 0.0020
Top-5 Accuracy: 0.0143
